# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing dataset elements by their `@id` as per the Croissant schema.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the Croissant-described dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and create a Dataset object (this loads schema only, not the data)
dataset = mlc.Dataset(croissant_url)

# Display the main dataset name and description
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, their `@id`s and a sample record. `mlcroissant` enables schema inspection before data loading.


In [ ]:
# List all record sets and their fields/columns using their @id
record_sets = dataset.metadata.record_sets
print("Available record sets and their fields (using @id):\n")
for rset in record_sets:
    print(f"- Record set name: {rset.name}, @id: {rset.id}")
    print("  Fields & columns:")
    for field in rset.fields:
        print(f"    • Field name: {field.name}, @id: {field.id}, column: {field.column}")
    print()

# Preview a single sample record by @id from the first record set
if record_sets:
    record_set_id = record_sets[0].id
    print(f"Sample record from record set {record_set_id}:")
    records = dataset.records(record_set=record_set_id)
    print(next(records))

## 3. Data Extraction

Load data from each record set into separate DataFrames, referencing each by their `@id`. We demonstrate with extraction for all record sets found in the Croissant schema.


In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Load all data into a dictionary of DataFrames {record_set_id: dataframe}
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for {rs_id}\n")

# Show columns for the first record set
if record_set_ids:
    first_id = record_set_ids[0]
    print(f"Columns in DataFrame for record set {first_id}:")
    print(dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)

Process, filter and analyze one of the numeric fields using only Croissant `@id` for reference. We'll:
- Select a numeric field from the first record set by its field `@id` (e.g., 'coverage' or 'MW').
- Filter for records with high values, normalize the field, and optionally group by a categorical field if present.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with the actual field `@id` from the overview above if you wish to explore a different field.

In [ ]:
# For this example, we select the following (update as needed based on the actual @id shown above):
# Suppose the first record set contains a numeric field with name "MW" (molecular weight); see its @id.

selected_record_set_id = record_set_ids[0]
df = dataframes[selected_record_set_id]

# Identify a numeric field by @id (list columns to confirm)
print(f"Columns in {selected_record_set_id}:", df.columns.tolist())

numeric_field_id = None
for field in dataset.metadata.record_sets[0].fields:
    if field.name.lower() in ["mw", "coverage", "peptides", "psms", "abundance", "normalized abundance"]:
        numeric_field_id = field.id
        print(f"Selected numeric field: {field.name} (id={field.id})")
        break

# If field not found, pick the first numeric column as fallback
if numeric_field_id is None:
    for c in df.columns:
        # Try to see if column is numeric (not NaN and convertible)
        try:
            df[c] = pd.to_numeric(df[c])
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                print(f"Fallback to first numeric field: {c}")
                break
        except Exception:
            continue

if numeric_field_id is not None:
    # Ensure column is float
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75)  # Use top 25% as an example threshold

    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (top 25%):")
    display(filtered_df.head())

    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Try to find a categorical/grouping field
    group_field_id = None
    for field in dataset.metadata.record_sets[0].fields:
        # Pick first non-numeric field
        if field.id != numeric_field_id and df[field.id].dtype == object:
            group_field_id = field.id
            print(f"Using group field: {field.name} (id={field.id})")
            break
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in {selected_record_set_id}")
    plt.show()

    # If grouping available, show boxplots
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(12, 6))
        to_plot = df[[group_field_id, numeric_field_id]].dropna()
        sns.boxplot(data=to_plot, x=group_field_id, y=numeric_field_id)
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

The `mlcroissant` library provides convenient loading, querying, and schema-compliant processing for FAIR^2 datasets. We loaded the dataset using only Croissant `@id` references, explored its schema, performed basic filtering and normalization on numeric fields, grouped by categorical attributes, and visualized data distributions.

For advanced analysis, consider further schema exploration, joining with other record sets, or applying ML pipelines directly to these DataFrames.
